# DSA with anchor writes, on CODI's training recipe

Trains one DSA arm that writes the anchor on every loop, then evaluates it. Recipe, data and seed
match the no-write arm in `docs/FINDINGS.md` section 5, so the two compare directly. That arm made no
anchor writes, so this is the first run on this recipe to test anchoring.

The baselines are not rerun. Their finished rows come from the repo and appear in the final table.

**Before running:**
- **Free at least 6 GB of Google Drive storage**, including its Trash, at
  https://drive.google.com/settings/storage. When Drive fills up, uploads stop without any error in
  the notebook, and that cost the last run its checkpoint. Each backup is about 1.2 GB.
- Runtime → Change runtime type → **GPU** (A100 if offered, otherwise T4), then Runtime → **Run all**.
- Keep this tab open until the last cell, which downloads the results to your computer.

Measured on an A100 for the no-write arm: about 42 minutes of training and 4 minutes of evaluation.
Writes may change that, so the notebook prints an upper bound before training. Evaluation data and the
prepared training rows are reused from Drive.

**If Colab disconnects:** reconnect to the same GPU type and Run all again. The run resumes from its
last Drive backup (about every 20 minutes) at the same code commit. A different GPU type cannot
resume a started run; set `RUN_TAG` to start a new one.

In [ ]:
# A run is identified by these settings; rerunning with the same values resumes it.
TRAIN_RECORDS = 32_768  # one pass at 128 records per update = 256 updates
WRITE_MODE = "every_loop"  # or "periodic", "learned", "static" (no anchor writes)
STEPS_JOIN = "space"    # "newline": one reasoning step per line, the format the checkpoint learned
NEW_MODULE_LR = None    # None: the anchor, workspace, controller and projection also use 3e-3
RUN_TAG = ""            # e.g. "-b" for a fresh run with the same settings
REPO_REF = "main"       # code for a new run; a started run always resumes at its own commit
assert TRAIN_RECORDS % 128 == 0, "use a multiple of 128 records"

In [ ]:
from pathlib import Path
import json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
RUN_NAME = f"codi-{WRITE_MODE}-{STEPS_JOIN}-n{TRAIN_RECORDS}{RUN_TAG}"
BACKUP_DIR = DRIVE / "codi-recipe" / RUN_NAME
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

STARTED = BACKUP_DIR / "backup/latest.json"
ref = json.loads((BACKUP_DIR / "plan.json").read_text())["commit"] if STARTED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print("Code:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(), f"(from {ref})")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = str(DRIVE / "cache/huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
RESULTS = DRIVE / "results"
for path in Path("results").rglob("*"):  # Seed Drive results with the committed rows for comparison.
    target = RESULTS / path.relative_to("results")
    if path.is_file() and not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
WORK = Path("/content/codi-recipe")  # fast local disk; checkpoints are backed up to BACKUP_DIR
WORK.mkdir(exist_ok=True)
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

def drive_out_of_storage():  # Drive keeps showing files it failed to upload, so check its own log.
    log = Path("/root/.config/Google/DriveFS/Logs/drive_fs.txt")
    return log.exists() and "QUOTA_EXCEEDED" in log.read_text(errors="replace")

In [ ]:
run("scripts/setup_data.py", "--eval-only")  # evaluation sets and the pinned GSM8K-Aug arrays
SMOKE = WORK / "smoke-cuda.json"
run("scripts/smoke_pretrained_dsa.py", "--device", "cuda", "--output", SMOKE, "--max-new-tokens", 4)
# A seeded sample of the training split, prepared once on Drive and reused afterwards.
PREPARED_DRIVE = DRIVE / "prepared" / f"codi-train-{TRAIN_RECORDS}-{STEPS_JOIN}"
run("scripts/prepare_dsa_pilot.py", "--source-dir", os.environ["DSA_DATA_DIR"], "--output-dir", PREPARED_DRIVE,
    "--train-limit", TRAIN_RECORDS, "--loops", 8, "--seed", 7, "--steps-join", STEPS_JOIN)
PREPARED = WORK / PREPARED_DRIVE.name
shutil.copytree(PREPARED_DRIVE, PREPARED, dirs_exist_ok=True)

In [ ]:
if drive_out_of_storage():
    print("WARNING: Drive is out of storage, so backups will not upload. The last cell still downloads the "
          "results, but a disconnect would lose the run. To be safe, stop here, free space and Run all again.")
# Picks a batch size that fits (first run only), trains with Drive backups, then evaluates
# 8, 4 and 1 loops and the explicit route on dev64 and the 500 validation questions.
NAME = f"dsa-codi-{WRITE_MODE}-n{TRAIN_RECORDS // 1024}k" + ("-newline" if STEPS_JOIN == "newline" else "") + RUN_TAG
run("scripts/run_codi_recipe.py", "--prepared-dir", PREPARED, "--work-dir", WORK / RUN_NAME,
    "--backup-dir", BACKUP_DIR, "--results-dir", RESULTS, "--smoke-report", SMOKE, "--device", "cuda",
    "--name", NAME, "--write-mode", WRITE_MODE,
    *([] if NEW_MODULE_LR is None else ["--new-module-learning-rate", NEW_MODULE_LR]))

In [ ]:
run("scripts/evaluate.py", "--table", "--results-dir", RESULTS)

In [ ]:
# Takes this run's rows straight from the session, so they survive even if Drive stopped syncing.
from google.colab import files

bundle = Path(f"/content/{NAME}")
shutil.rmtree(bundle, ignore_errors=True)
for kind in ("runs", "predictions"):
    for folder in (RESULTS / kind).glob(f"{NAME}-*"):
        shutil.copytree(folder, bundle / "results" / kind / folder.name)
training = bundle / "results/training" / NAME
training.mkdir(parents=True)
for path in (BACKUP_DIR / "plan.json", WORK / RUN_NAME / "metrics.jsonl"):
    shutil.copy2(path, training)
shutil.copy2(RESULTS / "leaderboard.csv", bundle / "results")
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip. If it does not start, get it from the Files panel on the left.")
if drive_out_of_storage():
    print("WARNING: Drive ran out of storage. Keep this zip: it may be the only copy of these results.")

**Reading the result**
- `-k8` against `-k1` is the first check, and the summary above gives the paired counts. The no-write
  arm scored 7.4% with 8 loops and 9.0% with 1 on validation (only 8 right 11, only 1 right 19,
  p = 0.20), and its answers had nearly settled by 4 loops.
- `-k8` against `dsa-codi-static-n32k-k8` shows what the writes add. On 500 questions, a difference of a
  couple of points can be noise.
- `-explicit` should stay near the no-write arm's 41.0% and `gpt2-sft`'s 42.4%.
- The DSA rows stay on dev64 and validation. The held-out sets are for its final numbers.

**Sharing:** the last cell downloads a zip laid out like the repo's `results/` folder. Point Claude at it.